# TradeStation SDK - Placing Orders

This notebook demonstrates how to:
- Place market orders
- Place limit orders
- Place stop orders
- Place trailing stop orders
- Cancel and modify orders
- Query order history and status

⚠️ **Important:** All examples use PAPER mode (simulator) for safety.

## Prerequisites

1. Complete `01_authentication.ipynb` first
2. Have valid TradeStation authentication (PAPER mode)
3. Understand order types and risks

In [ ]:
# Import libraries
from dotenv import load_dotenv
from tradestation_sdk import InvalidRequestError, TradeStationSDK

# Load environment variables
load_dotenv()

# Initialize and authenticate
sdk = TradeStationSDK()
sdk.ensure_authenticated(mode="PAPER")

# Get account info
account = sdk.get_account_info(mode="PAPER")
print(f"✅ Connected to account: {account['account_id']}")
print(f"📊 Account: {account['name']}")

# Check buying power
balances = sdk.get_account_balances(mode="PAPER")
print(f"💰 Buying Power: ${balances['buying_power']:,.2f}")

## 1. Place a Market Order

Market orders execute immediately at the current market price.

In [ ]:
# Check current position first
current_position = sdk.get_position("AAPL", mode="PAPER")
print(f"Current AAPL position: {current_position}")
print()

# Place market order (COMMENTED OUT FOR SAFETY)
# Uncomment to place actual order in PAPER mode

# order_id, status = sdk.place_order(
#     symbol="AAPL",
#     side="BUY",
#     quantity=10,
#     order_type="Market",
#     mode="PAPER"
# )
# print(f"✅ Market order placed: {order_id}")
# print(f"   Status: {status}")

print("ℹ️ Market order code commented out for safety")
print("ℹ️ Uncomment the code above to place actual order")

## 2. Place a Limit Order

Limit orders only execute at your specified price or better.

In [ ]:
# Get current price first
quotes = sdk.get_quote_snapshots("AAPL", mode="PAPER")
current_price = quotes["Quotes"][0].get("Last", 150.0)
print(f"Current AAPL price: ${current_price}")
print()

# Calculate limit price (below current for buy order)
limit_price = round(current_price * 0.99, 2)  # 1% below current
print(f"Limit price (1% below): ${limit_price}")
print()

# Place limit order (COMMENTED OUT FOR SAFETY)
# order_id, status = sdk.place_limit_order(
#     symbol="AAPL",
#     side="BUY",
#     quantity=10,
#     limit_price=limit_price,
#     time_in_force="DAY",
#     mode="PAPER"
# )
# print(f"✅ Limit order placed: {order_id}")
# print(f"   Limit Price: ${limit_price}")
# print(f"   Status: {status}")

print("ℹ️ Limit order code commented out for safety")

## 3. Place a Stop Order

Stop orders trigger when price reaches your stop price (used for stop-losses).

In [ ]:
# Stop price (below current for sell-stop)
stop_price = round(current_price * 0.95, 2)  # 5% below current
print(f"Stop price (5% below): ${stop_price}")
print()

# Place stop order (COMMENTED OUT FOR SAFETY)
# order_id, status = sdk.place_stop_order(
#     symbol="AAPL",
#     side="SELL",
#     quantity=10,
#     stop_price=stop_price,
#     mode="PAPER"
# )
# print(f"✅ Stop order placed: {order_id}")
# print(f"   Stop Price: ${stop_price}")

print("ℹ️ Stop order code commented out for safety")

## 4. Place a Trailing Stop Order

Trailing stops move with the price to lock in profits.

In [ ]:
# Trailing stop (trails by $3.00)
# For stocks: 1 point = $1.00, so trail_amount=3.0 means $3.00
# For MNQ futures: 1 point = $2.00, so trail_amount=1.5 means $3.00

# order_id, status = sdk.place_trailing_stop_order(
#     symbol="AAPL",
#     side="SELL",
#     quantity=10,
#     trail_amount=3.0,  # $3.00 trail
#     mode="PAPER"
# )
# print(f"✅ Trailing stop placed: {order_id}")
# print(f"   Trail Amount: $3.00")

print("ℹ️ Trailing stop code commented out for safety")
print("ℹ️ Note: trail_amount is in price points (stocks: 1 point = $1)")

## 5. Query Order History

Retrieve past orders to analyze trading activity.

In [ ]:
# Get order history (last 7 days, max 100 orders)
orders = sdk.get_order_history(start_date="2025-12-01", limit=100, mode="PAPER")

print(f"📝 Found {len(orders)} historical order(s)\n")

# Show recent orders
for order in orders[:5]:  # First 5
    order_id = order.get("OrderID", "N/A")
    symbol = order.get("Symbol", "N/A")
    status = order.get("Status", "N/A")
    quantity = order.get("Quantity", "N/A")
    filled = order.get("FilledQuantity", 0)

    print(f"Order {order_id}: {symbol} x {quantity} - Status: {status}")
    if filled:
        avg_price = order.get("AverageFillPrice", 0)
        print(f"  Filled: {filled} @ ${avg_price:.2f}")

## 6. Get Current/Open Orders

In [ ]:
# Get all current orders
current = sdk.get_current_orders(mode="PAPER")
orders = current.get("Orders", [])

print(f"📋 Current open orders: {len(orders)}\n")

if orders:
    for order in orders:
        print(f"Order {order['OrderID']}:")
        print(f"  Symbol: {order['Symbol']}")
        print(f"  Side: {order['TradeAction']}")
        print(f"  Type: {order['OrderType']}")
        print(f"  Quantity: {order['Quantity']}")
        print(f"  Status: {order['Status']}")
        print()
else:
    print("No open orders")

## 7. Cancel an Order

Cancel orders by order ID (from order placement or query).

In [ ]:
# Get open orders to cancel
current = sdk.get_current_orders(mode="PAPER")
orders = current.get("Orders", [])

if orders:
    # Cancel first order (COMMENTED OUT FOR SAFETY)
    # order_id = orders[0]['OrderID']
    # success, message = sdk.cancel_order(order_id, mode="PAPER")
    #
    # if success:
    #     print(f"✅ Order {order_id} cancelled")
    # else:
    #     print(f"❌ Cancellation failed: {message}")

    print(f"ℹ️ Found {len(orders)} order(s) to cancel")
    print("ℹ️ Cancel code commented out for safety")
else:
    print("No orders to cancel")

## 8. Error Handling Example

Demonstrate proper error handling for order placement.

In [ ]:
import time

from tradestation_sdk import AuthenticationError, RateLimitError, TradeStationAPIError


def place_order_safe(symbol, side, quantity, **kwargs):
    """Place order with error handling and retry logic."""
    max_retries = 3

    for attempt in range(max_retries):
        try:
            order_id, status = sdk.place_order(symbol=symbol, side=side, quantity=quantity, mode="PAPER", **kwargs)
            print(f"✅ Order placed: {order_id}")
            return order_id, status

        except AuthenticationError:
            print("🔐 Re-authenticating...")
            sdk.authenticate(mode="PAPER")
            continue

        except RateLimitError:
            if attempt < max_retries - 1:
                wait = 2**attempt
                print(f"⏳ Rate limit hit. Waiting {wait}s...")
                time.sleep(wait)
                continue
            raise

        except InvalidRequestError as e:
            print(f"❌ Invalid request: {e}")
            return None, str(e)

        except TradeStationAPIError as e:
            print(f"❌ API error: {e}")
            return None, str(e)

    return None, "Max retries exceeded"


# Test error handling (uncomment to test)
# result = place_order_safe("AAPL", "BUY", 10, order_type="Market")
print("✅ Error handling function defined")
print("ℹ️ Uncomment to test order placement with error handling")

## Summary

You've learned how to:
- ✅ Place market, limit, stop, and trailing stop orders
- ✅ Query order history and current orders
- ✅ Cancel orders
- ✅ Handle errors properly with retry logic

## Important Notes

- Always use PAPER mode for testing
- Check positions before placing orders
- Implement error handling and retries
- Monitor order status after placement
- Be aware of API rate limits (60 orders/minute)

## Next Steps

- 📊 [05_streaming_data.ipynb](05_streaming_data.ipynb) - Real-time data streaming
- 🎯 [06_bracket_orders.ipynb](06_bracket_orders.ipynb) - Advanced bracket orders
- 📍 [07_position_management.ipynb](07_position_management.ipynb) - Position tracking